In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as an
import sys
sys.path.append('/home/jovyan/notebooks_bidossessi/Inflammatory/Remapped/autoimmuneSC/AIID_notebooks/')
from inflapy.immunoFunc import *

In [2]:

s_path = "/nfs/team205/bh14/Datasets/Remapped/raw_adata/ummapped/"

In [23]:
#### Concat

In [4]:
os.listdir(s_path+"Tissue/QC_Doublet/")

['Martin_JIA_Synovial_fluid',
 'Qian_Yue_Zhang_2022.h5ad',
 'Martin_JIA_Synovial_fluid.h5ad',
 'AMP_phase1_RA',
 'Kong_CD_2023']

In [6]:
adatas = ['Qian_Yue_Zhang_2022.h5ad','Martin_JIA_Synovial_fluid.h5ad','AMP_phase1_RA', 'Kong_CD_2023']

In [7]:
path = s_path+"Tissue/QC_Doublet/"
concat_info=[]

for i in range(len(adatas)):
      
    print(str(i)+': reading and concatenating '+adatas[i]+' ...')
        
    adata = sc.read_h5ad(path+adatas[i])
    print(adata.shape)
    
    if 'adata_concat' in locals():
        
        adata_concat.obs_names_make_unique()
        #adata_concat.var_names_make_unique()
        #adata_concat = ad.concat([adata_concat, adata], join="inner", index_unique="_")
        adata_concat = adata_concat.concatenate(adata, join="inner", index_unique="_")
        
    else:
        adata_concat = adata
    
    print(adata_concat.shape)
    mydict = {'Dataset':adatas[i],'Tissue':adata.obs['tissue'].unique().to_list(),
              'Cell_num':adata.shape[0], 'Gene_num':adata.shape[1], 'gene_after_concat':adata_concat.shape[1]
             }
    a_mydict = mydict.copy()
    #print(a_mydict)
    #I used .copy() and ignore_index=True to avoid error when appending dict to list
    concat_info.append(a_mydict.copy())
    #concat_info = pd.json_normalize(concat_info)
    # Show QC of each dataset
    #showAdataQc(adata=adata,file=files[i])
    del adata
    gc.collect()
    #adata_concat = sc.concat(adatas, join="inner", index_unique="_", uns_merge='unique')

0: reading and concatenating Qian_Yue_Zhang_2022.h5ad ...
(56199, 33694)
(56199, 33694)
1: reading and concatenating Martin_JIA_Synovial_fluid.h5ad ...
(19949, 33694)
(76148, 33645)
2: reading and concatenating AMP_phase1_RA ...
(10099, 29420)
(86247, 24760)
3: reading and concatenating Kong_CD_2023 ...
(353581, 27830)
(439828, 23044)


In [10]:
concat_info

[{'Dataset': 'Qian_Yue_Zhang_2022.h5ad',
  'Tissue': ['Thyroid'],
  'Cell_num': 56199,
  'Gene_num': 33694,
  'gene_after_concat': 33694},
 {'Dataset': 'Martin_JIA_Synovial_fluid.h5ad',
  'Tissue': ['Synovial fluid'],
  'Cell_num': 19949,
  'Gene_num': 33694,
  'gene_after_concat': 33645},
 {'Dataset': 'AMP_phase1_RA',
  'Tissue': ['Synovial tissue'],
  'Cell_num': 10099,
  'Gene_num': 29420,
  'gene_after_concat': 24760},
 {'Dataset': 'Kong_CD_2023',
  'Tissue': ['colon', 'ileum'],
  'Cell_num': 353581,
  'Gene_num': 27830,
  'gene_after_concat': 23044}]

In [11]:
adata_concat

AnnData object with n_obs × n_vars = 439828 × 23044
    obs: 'tissue', 'disease', 'donor_id', 'dataset_id', 'cell_source', 'doublet_scores', 'predicted_doublets', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'QC', 'sex', 'development_stage', 'batch', 'orig.ident', 'Site'
    var: 'mt', 'rb', 'n_cells_by_counts-0-0-0', 'mean_counts-0-0-0', 'pct_dropout_by_counts-0-0-0', 'total_counts-0-0-0', 'n_cells_by_counts-1-0-0', 'mean_counts-1-0-0', 'pct_dropout_by_counts-1-0-0', 'total_counts-1-0-0', 'n_cells_by_counts-1-0', 'mean_counts-1-0', 'pct_dropout_by_counts-1-0', 'total_counts-1-0', 'n_cells_by_counts-1', 'mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1'

In [12]:
def filter_norma(adata, min_genes=500, min_donor=50):
    #sc.pp.filter_cells(adata, min_counts=min_counts)

    sc.pp.filter_cells(adata, min_genes=min_genes)

    adata = adata[adata.obs['QC'] == 'Pass']

    # Remove donors with few than 50 cells
    donor_counts = adata.obs['donor_id'].value_counts()

    donor_counts.sort_values()[:30]

    donor_counts[donor_counts<min_donor].sum()

    adata = adata[adata.obs['donor_id'].isin( donor_counts[donor_counts>=min_donor].index)]
    
    adata.layers["counts"] = adata.X.copy()
# size and log1p normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    adata.raw = adata  # keep full dimension safe
    return adata

In [13]:
adata_concat = filter_norma(adata=adata_concat)

In [14]:
adata_concat.obs[['dataset_id', 'tissue']].value_counts()

dataset_id           tissue         
Kong_et_al_2023      ileum              114772
                     colon               97539
Qian_Yue_Zhang_2022  Thyroid             47781
Martin_JIA_2019      Synovial fluid      19933
AMP_phase1_RA        Synovial tissue      4001
dtype: int64

In [15]:
pd.DataFrame(concat_info)

,Dataset,Tissue,Cell_num,Gene_num,gene_after_concat
0,Qian_Yue_Zhang_2022.h5ad,[Thyroid],56199,33694,33694
1,Martin_JIA_Synovial_fluid.h5ad,[Synovial fluid],19949,33694,33645
2,AMP_phase1_RA,[Synovial tissue],10099,29420,24760
3,Kong_CD_2023,"[colon, ileum]",353581,27830,23044


In [16]:
adata_concat.write_h5ad("/nfs/team205/bh14/Datasets/Remapped/raw_adata/ummapped/Tissue/QC_Doublet_concat/adata_query_tissue_2.h5ad")


In [18]:
adata_concat.X.expm1().sum(axis = 1)

matrix([[ 9999.999],
        [ 9999.999],
        [10000.   ],
        ...,
        [10000.   ],
        [10000.   ],
        [10000.001]], dtype=float32)